# P12 Target-Clue Leakage Reproducible

由原 P12 拆出：檢查生成描述是否直接包含目標單品線索。


## 0. Setup


In [1]:
from __future__ import annotations

from pathlib import Path
from statistics import mean, median, stdev
from collections import Counter, defaultdict
from datetime import datetime
import csv
import json
import math
import re

try:
    from scipy import stats as scipy_stats
    SCIPY_AVAILABLE = True
except Exception:
    scipy_stats = None
    SCIPY_AVAILABLE = False

BASE = Path("/home/ester/valid_description")
TGT = BASE / "experiment data" / "journal_figures_tables_20260629"
FIG = TGT / "figures"
TAB = TGT / "tables"
DER = TGT / "derived_audits"
SRC = TGT / "source_programs"
REPO = BASE / "text-conditioned-outfit-recommendation"
EXP_MAIN = REPO / "experiments"
EXP_NONE = REPO / "experiments_none_only"

for p in [FIG, TAB, DER, SRC]:
    p.mkdir(parents=True, exist_ok=True)

TOKEN_RE = re.compile(r"[A-Za-z0-9]+(?:[-'][A-Za-z0-9]+)?")
WORD_RE = re.compile(r"[A-Za-z]+(?:[-'][A-Za-z]+)?")

def read_json(path: Path):
    with path.open(encoding="utf-8") as f:
        return json.load(f)

def read_csv(path: Path) -> list[dict[str, str]]:
    with path.open(newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))

def write_csv(path: Path, rows: list[dict]) -> None:
    if not rows:
        raise ValueError(f"No rows to write: {path}")
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    print("[write]", path)

def display_csv(path: Path, max_rows: int = 12) -> None:
    rows = read_csv(path)
    for r in rows[:max_rows]:
        print(r)
    if len(rows) > max_rows:
        print(f"... ({len(rows) - max_rows} more rows)")

def fmt_float(x, nd=4) -> str:
    try:
        if x is None or math.isnan(float(x)):
            return ""
        return f"{float(x):.{nd}f}"
    except Exception:
        return ""

def fmt_pct(num, den) -> str:
    return f"{100 * float(num) / float(den):.2f}" if den else "0.00"

def to_int(x, default=0) -> int:
    try:
        return int(float(x))
    except Exception:
        return default

def to_float(x, default=0.0) -> float:
    try:
        return float(x)
    except Exception:
        return default

def query_token_count(text: str) -> int:
    return len(TOKEN_RE.findall(str(text or "").strip()))

def word_tokens(text: str) -> list[str]:
    return [t.lower() for t in WORD_RE.findall(str(text or "").strip())]

def word_count(text: str) -> int:
    return len(word_tokens(text))

def original_field(original_data: dict, set_id: str, field: str) -> str:
    value = original_data.get(str(set_id), "")
    if isinstance(value, dict):
        return str(value.get(field, "") or "").strip()
    return str(value or "").strip()

def original_url_name_plus_title(original_data: dict, set_id: str) -> str:
    value = original_data.get(str(set_id), "")
    if isinstance(value, dict):
        parts = [str(value.get("url_name", "") or "").strip(), str(value.get("title", "") or "").strip()]
        return " ".join(p for p in parts if p).strip()
    return str(value or "").strip()

def generated_title_only(generated_data: dict, set_id: str) -> str:
    value = generated_data.get(str(set_id), "")
    if isinstance(value, dict):
        return str(value.get("title", "") or "").strip()
    return str(value or "").strip()

print("BASE =", BASE)
print("TAB  =", TAB)
print("DER  =", DER)
print("SCIPY_AVAILABLE =", SCIPY_AVAILABLE)


BASE = /home/ester/valid_description
TAB  = /home/ester/valid_description/experiment data/journal_figures_tables_20260629/tables
DER  = /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits
SCIPY_AVAILABLE = True


## 3. Target-Clue Leakage Audit


In [4]:
STOP = set("""
a an and are as at be by for from in into is it its of on or the this that these those to with without your you
women woman womens female fashion clothing clothes outfit outfits wear wearing style styles styled look looks item items piece pieces
new used vintage size small medium large plus petite one pair set collection design designer brand brands online shop shopping
classic modern casual chic sleek elegant effortless comfy cozy cool warm cold hot day days night nights spring summer fall autumn winter
vibe vibes scene mood ready friendly perfect practical realistic recommended temperature original title url name
""".split())
WORD_RE2 = re.compile(r"[a-zA-Z][a-zA-Z'-]*")

def words(s):
    return [w.strip("'-").lower() for w in WORD_RE2.findall(str(s or "").lower()) if w.strip("'-")]

def category_match(text, fine, major):
    text_words = set(words(text))
    text_phrase = " ".join(words(text))
    def match_label(label):
        ws = words(label)
        if not ws: return False
        return " ".join(ws) in text_phrase or any(w in text_words for w in ws)
    return match_label(fine), match_label(major)

def meta_parts(meta):
    return [str(meta.get(k, "") or "").strip() for k in ["url_name", "title", "description", "semantic_category"] if str(meta.get(k, "") or "").strip()]

def distinctive_tokens(meta, fine, major):
    blocked = set(words(fine)) | set(words(major)) | STOP
    toks = []
    for part in meta_parts(meta):
        toks.extend(w for w in words(part) if len(w) >= 4 and w not in blocked)
    return sorted(set(toks))

def distinctive_bigrams(meta, fine, major):
    toks = []
    for part in meta_parts(meta):
        ws = [w for w in words(part) if w not in STOP and len(w) >= 3]
        toks.extend(" ".join(ws[i:i+2]) for i in range(len(ws)-1))
    blocked = {" ".join(words(fine)), " ".join(words(major))}
    return sorted({b for b in toks if b and b not in blocked})

rel_path = REPO / "subset_analysis_results/reliability_meta_from_subset.csv"
rel_rows = read_csv(rel_path)
item_meta = read_json(BASE / "polyvore_data/polyvore_outfits/polyvore_item_metadata.json")
original_titles = read_json(BASE / "polyvore_data/polyvore_outfits/polyvore_outfit_titles.json")
generated_titles = read_json(BASE / "new_polyvore_outfit_titles.json")
unique = {}
for r in rel_rows:
    unique.setdefault((str(r["set_id"]), str(r["target_item_id"])), r)

audited = []
for (sid, iid), r in sorted(unique.items(), key=lambda x: (int(x[0][0]), int(x[0][1]))):
    fine = str(r.get("fine_category", "") or "").lower()
    major = str(r.get("major_category", "") or "").lower()
    gen_text = str(r.get("title_full_text", "") or "").strip() or generated_title_only(generated_titles, sid)
    orig_text = str(r.get("original_text", "") or "").strip() or original_url_name_plus_title(original_titles, sid)
    meta = item_meta.get(str(iid), {})
    dtoks, dbigrams = distinctive_tokens(meta, fine, major), distinctive_bigrams(meta, fine, major)
    gen_words, orig_words = set(words(gen_text)), set(words(orig_text))
    gen_phrase, orig_phrase = " ".join(words(gen_text)), " ".join(words(orig_text))
    gf, gm = category_match(gen_text, fine, major)
    of, om = category_match(orig_text, fine, major)
    gen_tok = sorted([t for t in dtoks if t in gen_words])
    gen_big = sorted([b for b in dbigrams if b in gen_phrase])
    orig_tok = sorted([t for t in dtoks if t in orig_words])
    orig_big = sorted([b for b in dbigrams if b in orig_phrase])
    audited.append({
        "set_id": sid, "target_item_id": iid, "fine_category": fine, "major_category": major,
        "generated_text": gen_text, "original_text": orig_text,
        "target_item_title_or_url": str(meta.get("title", "") or meta.get("url_name", "") or ""),
        "generated_fine_category_match": str(gf), "generated_any_category_match": str(gf or gm),
        "generated_target_token_overlap_count": str(len(gen_tok)), "generated_target_token_overlap": " | ".join(gen_tok[:12]),
        "generated_target_bigram_overlap_count": str(len(gen_big)), "generated_target_bigram_overlap": " | ".join(gen_big[:12]),
        "original_fine_category_match": str(of), "original_any_category_match": str(of or om),
        "original_target_token_overlap_count": str(len(orig_tok)), "original_target_token_overlap": " | ".join(orig_tok[:12]),
        "original_target_bigram_overlap_count": str(len(orig_big)), "original_target_bigram_overlap": " | ".join(orig_big[:12]),
    })

n = len(audited)
def count(pred): return sum(1 for a in audited if pred(a))
a03 = [
    {"metric": "audited_unique_main_cir_queries", "count": str(n), "rate_pct": "100.00", "interpretation": "Main CIR query-target pairs, not no_missing subset."},
    {"metric": "generated_fine_category_lexical_match", "count": str(count(lambda a: a["generated_fine_category_match"] == "True")), "rate_pct": fmt_pct(count(lambda a: a["generated_fine_category_match"] == "True"), n), "interpretation": "Static category-token check; category mentions can be semantically natural and are not automatically answer leakage."},
    {"metric": "generated_any_category_lexical_match", "count": str(count(lambda a: a["generated_any_category_match"] == "True")), "rate_pct": fmt_pct(count(lambda a: a["generated_any_category_match"] == "True"), n), "interpretation": "Fine or major category mention in generated query."},
    {"metric": "generated_target_item_distinctive_token_overlap", "count": str(count(lambda a: to_int(a["generated_target_token_overlap_count"]) > 0)), "rate_pct": fmt_pct(count(lambda a: to_int(a["generated_target_token_overlap_count"]) > 0), n), "interpretation": "Potential direct target-item lexical clue; inspect examples."},
    {"metric": "generated_target_item_distinctive_bigram_overlap", "count": str(count(lambda a: to_int(a["generated_target_bigram_overlap_count"]) > 0)), "rate_pct": fmt_pct(count(lambda a: to_int(a["generated_target_bigram_overlap_count"]) > 0), n), "interpretation": "Stricter phrase-level clue check."},
    {"metric": "original_target_item_distinctive_token_overlap", "count": str(count(lambda a: to_int(a["original_target_token_overlap_count"]) > 0)), "rate_pct": fmt_pct(count(lambda a: to_int(a["original_target_token_overlap_count"]) > 0), n), "interpretation": "Baseline reference: original url_name + title can also contain item-like tokens."},
]
write_csv(DER / "A03_target_item_clue_leakage_audit_summary.csv", a03)
examples = [a for a in audited if to_int(a["generated_target_token_overlap_count"]) > 0 or to_int(a["generated_target_bigram_overlap_count"]) > 0]
examples = sorted(examples, key=lambda a: (to_int(a["generated_target_bigram_overlap_count"]), to_int(a["generated_target_token_overlap_count"])), reverse=True)[:80]
write_csv(DER / "A04_target_item_clue_leakage_examples.csv", examples if examples else audited[:1])
display_csv(DER / "A03_target_item_clue_leakage_audit_summary.csv")


[write] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A03_target_item_clue_leakage_audit_summary.csv
[write] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A04_target_item_clue_leakage_examples.csv
{'metric': 'audited_unique_main_cir_queries', 'count': '9311', 'rate_pct': '100.00', 'interpretation': 'Main CIR query-target pairs, not no_missing subset.'}
{'metric': 'generated_fine_category_lexical_match', 'count': '419', 'rate_pct': '4.50', 'interpretation': 'Static category-token check; category mentions can be semantically natural and are not automatically answer leakage.'}
{'metric': 'generated_any_category_lexical_match', 'count': '454', 'rate_pct': '4.88', 'interpretation': 'Fine or major category mention in generated query.'}
{'metric': 'generated_target_item_distinctive_token_overlap', 'count': '1030', 'rate_pct': '11.06', 'interpretation': 'Potential direct target-item lexical clue; insp